<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 04 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">迟到订单 Incremental</div>
  <p class="doris-cover-lead">模拟订单更正晚于首次加载到达，通过增量 merge 更新订单当前版本并验证幂等性。</p>
  <span class="doris-cover-note">Incremental · merge · Unique Key · on_schema_change · Data Test</span>
</div>

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 dbt-for-apache-doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 4：迟到订单 Incremental

这个 Demo 分阶段展示同一张订单源表发生变化后，dbt-doris 如何用 Unique Key + `merge` 更新当前版本。

<div class="doris-flow">
  <div class="doris-flow-step"><strong>初始事件</strong>3 条订单事件</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>版本识别</strong><code>order_version_history</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>首次写入</strong>Unique Key Incremental Table</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>迟到更新</strong>新版本 101 + 新订单 104</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Merge 结果</strong>保留 4 个订单当前版本</div>
</div>

### 2.1 准备并查看初始订单事件

源表按事件保存订单。订单 101、102、103 各有一个事件，后面会再写入订单 101 的新版本和订单 104。

In [ ]:
incremental_dir = runner.examples_root / "doris-demos/incremental"
runner.show_file("Fixture SQL", incremental_dir / "scripts/setup.sql")
runner.show_file("Source 声明", incremental_dir / "models/sources.yml")
runner.run_sql_file("创建 Incremental 源表", incremental_dir / "scripts/setup.sql")
runner.query("输入：初始订单事件", """
select event_id, order_id, customer_id, channel_id, grand_total, ordered_at, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 2.2 首次构建版本历史和 Incremental 表

`order_version_history` 用窗口函数给每个订单排序；`incremental_daily_sales` 只输出 `is_current = 1` 的版本，并配置 `unique_key='order_id'` 和 `incremental_strategy='merge'`。

In [ ]:
runner.show_file("版本历史 Model", incremental_dir / "models/order_version_history.sql")
runner.show_file("Incremental Model", incremental_dir / "models/incremental_daily_sales.sql")
runner.show_file("Incremental Test 定义", incremental_dir / "models/incremental.yml")
runner.run_dbt("首次全量构建", incremental_dir, "build")
runner.query("中间结果：当前订单版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")

### 2.3 写入迟到版本和新订单

新事件把订单 101 的金额从 100.00 改为 125.00，并新增订单 104。此时目标表还没有变化，变化只发生在源事件表。

In [ ]:
late_events_sql = """
insert into dbt_demo_incremental_source.ORDERS values
    (4, 101, 1, 'web', 125.00, 'COMPLETED', '2026-08-01 09:00:00', '2026-08-05 09:00:00'),
    (5, 104, 3, 'mobile', 70.00, 'COMPLETED', '2026-08-01 12:00:00', '2026-08-05 10:00:00')
"""
runner.show_sql("迟到事件 SQL", late_events_sql)
runner.run_sql("写入迟到版本和新订单", late_events_sql)
runner.query("变更后的源事件", """
select event_id, order_id, grand_total, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 2.4 执行增量 merge

第二次 `dbt build` 重新计算依赖 Model，Incremental Model 通过 `order_id` 合并新旧记录：101 变为版本 2，104 成为版本 1。

In [ ]:
runner.run_dbt("执行增量 merge", incremental_dir, "build")
runner.query("merge 后的当前版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")
runner.query("merge 后的每日收入", """
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date
""")

### 2.5 再次运行确认幂等性

源表没有新增数据时再次执行相同的 build，结果仍应保持 4 个订单和 245.00 的 8 月 1 日收入。

In [ ]:
runner.run_dbt("重复构建 Incremental Model", incremental_dir, "build")
runner.run_script("校验 Incremental Demo", incremental_dir / "scripts/verify.sh")
runner.query("幂等运行后的结果", """
select count(*) as order_rows, count(distinct order_id) as distinct_orders,
       sum(case when order_date = '2026-08-01' then grand_total else 0 end) as aug_01_revenue
from dbt_demo_incremental.incremental_daily_sales
""")

## 完成

首次构建、迟到更新、Unique Key merge 和无变化重跑均已通过校验。